optional but is just cool

Load pretrained ResNet18 or MobileNetV2
Adapt for 24-class classification
Fine-tune model
Compare to custom CNN
Get best accuracy (~90-93%)

imports and loading 

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)


In [7]:
#load data
train_data = pd.read_csv('../data/sign_mnist_train.csv')
test_data = pd.read_csv('../data/sign_mnist_test.csv')

X_train_full = train_data.drop('label', axis=1).values
y_train_full = train_data['label'].values

# Normalize
X_train_full = X_train_full / 255.0

#imp!!  MobileNet expects 3-channel RGB images
# We have grayscale, so we'll repeat the channel 3 times
import tensorflow as tf
X_train_full = X_train_full.reshape(-1, 28, 28, 1)
X_train_full = tf.image.resize(X_train_full, [32, 32]).numpy()
X_train_full = np.repeat(X_train_full, 3, axis=-1)  # grayscale to RGB

#one hot encode
y_train_full_encoded = keras.utils.to_categorical(y_train_full, 25)

#split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

print(f"\nData loaded and preprocessed!")
print(f"training set: {X_train.shape[0]:,} images")
print(f"Validation set: {X_val.shape[0]:,} images")
print(f"Image shape: {X_train.shape[1:]} (28x28x3 RGB)")


Data loaded and preprocessed!
training set: 21,964 images
Validation set: 5,491 images
Image shape: (32, 32, 3) (28x28x3 RGB)


In [11]:
print("\nUsing MobileNetV2 pretrained on ImageNet")
print("Adapting for 32x32x3 images and 25 classes")

#load pretrained MobileNetV2 (without top classification layer)
base_model = MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights=None  #cause we can't use ImageNet weights for 28x28
)

#freeze base model initially
base_model.trainable = False

#build complete model
transfer_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    
    #pretrained base
    base_model,
    
    #custom classification head
    layers.GlobalAveragePooling2D(name='gap'),
    layers.BatchNormalization(name='bn1'),
    layers.Dropout(0.5, name='dropout1'),
    
    layers.Dense(128, activation='relu', name='dense1'),
    layers.BatchNormalization(name='bn2'),
    layers.Dropout(0.3, name='dropout2'),
    
    layers.Dense(25, activation='softmax', name='output')
], name='transfer_learning_model')

transfer_model.summary()

total_params = transfer_model.count_params()
print(f"\nTotal parameters: {total_params:,}")


Using MobileNetV2 pretrained on ImageNet
Adapting for 32x32x3 images and 25 classes


Model: "transfer_learning_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_32             │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 1280)           │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 25)             │         3,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,430,809 (9.27 MB)

 Trainable params: 170,009 (664.10 KB)

 Non-trainable params: 2,260,800 (8.62 MB)


Total parameters: 2,430,809


### results 

Using MobileNetV2 pretrained on ImageNet
Adapting for 32x32x3 images and 25 classes
Model: "transfer_learning_model"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_32             │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 1280)           │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 25)             │         3,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 2,430,809 (9.27 MB)
 Trainable params: 170,009 (664.10 KB)
 Non-trainable params: 2,260,800 (8.62 MB)

Total parameters: 2,430,809

In [ ]:
# compile and train